# minibatch12 / comp3units — detection visualization

**Layouts.**
1. *Aggregated* — one row per test case, 3 sensors as columns; detections deduped over the whole stream.
2. *Per-timeslice* — one figure per test case; rows = aligned telemetry frames over time, cols = 3 sensors.

## Dataset

Source: `comp3units/<case>/{<sensor>.jpg, logs.txt}` — 6 cases × 3 sensors. Images are 320×240, calibrated mount height 2690 mm (from `info.txt`).

| Sensor UUID                  | Type                  |
| ---------------------------- | --------------------- |
| `6000099-1221-012345-4040`   | fixed **old** camera  |
| `6000015-1221-012345-4040`   | FP **new** camera (rotated)    |
| `6000037-1221-012345-4040`   | fixed **new** camera (rotated) |

## Log format

Each detection event looks like:

```
May 20 09:32:09 DUSUN pg2gw[141971]: Detection translated for sensor 6000099-1221-012345-4040: pixel(102,78) -> cm(-116,93). conf = 0.500000; box: x = 61, y = 33, w = 62, h = 60
```

A frame is delimited by `Received telemetry data, ... from thing <sensor>`. All detection lines that follow with the same timestamp/sensor belong to that frame (a frame may have 0, 1, or several detections).

Sensors don't report on the same cadence (037+099 fire together every ~10 s; 015 is offset by ~8 s), so we **bucket frames into timeslices** using a configurable time window (default 6 s). A timeslice holds at most one frame per sensor.

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

ROOT = Path('comp3units')

SENSORS = [
    ('6000099-1221-012345-4040', 'fixed old'),
    ('6000015-1221-012345-4040', 'FP new (rot)'),
    ('6000037-1221-012345-4040', 'fixed new (rot)'),
]

CASES = sorted([p.name for p in ROOT.iterdir() if p.is_dir()])
CASES

## 1 — Aggregated view (deduped across full stream)

Box label is `conf (xN)` where N is how many telemetry frames reported the exact same `(x,y,w,h)`. A stable real detection has high N; a one-off `x1` is likely noise.

In [ ]:
DET_RE = re.compile(
    r'Detection translated for sensor (?P<sensor>[\w-]+):\s*'
    r'pixel\((?P<px>-?\d+),(?P<py>-?\d+)\)\s*->\s*'
    r'cm\((?P<cx>-?\d+),(?P<cy>-?\d+)\)\.\s*'
    r'conf\s*=\s*(?P<conf>[\d.]+);\s*'
    r'box:\s*x\s*=\s*(?P<x>-?\d+),\s*y\s*=\s*(?P<y>-?\d+),\s*'
    r'w\s*=\s*(?P<w>\d+),\s*h\s*=\s*(?P<h>\d+)'
)

def parse_logs(log_path):
    """Aggregated: {sensor: [{x,y,w,h,conf,count,px,py}, ...]} deduped by (x,y,w,h)."""
    buckets = defaultdict(dict)
    for line in log_path.read_text().splitlines():
        m = DET_RE.search(line)
        if not m:
            continue
        s = m.group('sensor')
        box = (int(m['x']), int(m['y']), int(m['w']), int(m['h']))
        conf = float(m['conf'])
        prev = buckets[s].get(box)
        if prev is None:
            buckets[s][box] = dict(
                x=box[0], y=box[1], w=box[2], h=box[3],
                conf=conf, count=1,
                px=int(m['px']), py=int(m['py']),
            )
        else:
            prev['conf'] = max(prev['conf'], conf)
            prev['count'] += 1
    return {s: list(d.values()) for s, d in buckets.items()}

In [ ]:
def conf_color(conf):
    """Red < 0.5 <= orange < 0.7 <= lime."""
    if conf < 0.5:
        return '#ff3030'
    if conf < 0.7:
        return '#ff9f1c'
    return '#2ecc40'

def draw_detections(ax, img_path, detections, title, show_count=True):
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for det in detections:
        c = conf_color(det['conf'])
        ax.add_patch(Rectangle(
            (det['x'], det['y']), det['w'], det['h'],
            linewidth=1.6, edgecolor=c, facecolor='none',
        ))
        ax.plot(det['px'], det['py'], marker='+', color=c, markersize=8, mew=1.4)
        label = f"{det['conf']:.2f}" + (f" (x{det['count']})" if show_count and 'count' in det else '')
        ax.text(
            det['x'] + 2, max(det['y'] - 3, 8), label,
            fontsize=7, color='white',
            bbox=dict(facecolor=c, edgecolor='none', pad=1, alpha=0.85),
        )

In [ ]:
rows, cols = len(CASES), len(SENSORS)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * 3.3))
if rows == 1:
    axes = [axes]

for r, case in enumerate(CASES):
    case_dir = ROOT / case
    dets_by_sensor = parse_logs(case_dir / 'logs.txt')
    for c, (sensor, label) in enumerate(SENSORS):
        ax = axes[r][c]
        img_path = case_dir / f'{sensor}.jpg'
        dets = dets_by_sensor.get(sensor, [])
        title = f'{case} | {label}\n{sensor[:7]}  |  {len(dets)} det'
        draw_detections(ax, img_path, dets, title)

fig.suptitle('comp3units — aggregated detections per case (rows) × sensor (cols)', fontsize=12, y=1.005)
fig.tight_layout()
plt.show()

## 2 — Per-timeslice view

Each `Received telemetry data` line starts a new frame for that sensor; subsequent detection lines (same timestamp + sensor) attach to it. Empty frames (no detections) are kept — that's a true negative report.

Frames are then bucketed into timeslices with a `±window/2` tolerance so the three sensors' offset cadences line up into shared rows. If a sensor didn't report inside a slice, that cell is left blank with an `—` marker.

In [ ]:
TS_RE = re.compile(r'^\w+\s+\d+\s+(\d{2}):(\d{2}):(\d{2})\s')
TELE_RE = re.compile(r'Received telemetry data,[^\n]*from thing (?P<sensor>[\w-]+)')

def _ts_sec(line):
    m = TS_RE.match(line)
    if not m:
        return None
    h, mn, s = map(int, m.groups())
    return h * 3600 + mn * 60 + s

def parse_frames(log_path):
    """Return list of frames: [{ts, sensor, dets:[...]}, ...] in log order."""
    frames = []
    cur = None
    for line in log_path.read_text().splitlines():
        ts = _ts_sec(line)
        if ts is None:
            continue
        tele = TELE_RE.search(line)
        if tele:
            if cur is not None:
                frames.append(cur)
            cur = {'ts': ts, 'sensor': tele.group('sensor'), 'dets': []}
            continue
        det = DET_RE.search(line)
        if det and cur is not None and det.group('sensor') == cur['sensor']:
            cur['dets'].append(dict(
                x=int(det['x']), y=int(det['y']),
                w=int(det['w']), h=int(det['h']),
                conf=float(det['conf']),
                px=int(det['px']), py=int(det['py']),
            ))
    if cur is not None:
        frames.append(cur)
    return frames

def fmt_clock(ts):
    return f'{ts//3600:02d}:{(ts%3600)//60:02d}:{ts%60:02d}'

def align_timeslices(frames, window=6):
    """Greedy time-bucketing: a frame joins a slice if |ts - slice.ts| <= window
    and that sensor is not already filled. Slice ts = mean of joined frames.
    """
    slices = []
    for f in sorted(frames, key=lambda x: x['ts']):
        placed = False
        for slc in slices:
            if abs(slc['ts'] - f['ts']) <= window and f['sensor'] not in slc['by_sensor']:
                slc['by_sensor'][f['sensor']] = f['dets']
                n = len(slc['by_sensor'])
                slc['ts'] = (slc['ts'] * (n - 1) + f['ts']) / n
                placed = True
                break
        if not placed:
            slices.append({'ts': f['ts'], 'by_sensor': {f['sensor']: f['dets']}})
    return slices

# sanity check
_frames = parse_frames(ROOT / CASES[0] / 'logs.txt')
_slices = align_timeslices(_frames)
print(f'{CASES[0]}: {len(_frames)} frames -> {len(_slices)} timeslices')
for s in _slices[:3]:
    print(' ', fmt_clock(int(s['ts'])), {k[:7]: len(v) for k, v in s['by_sensor'].items()})

In [ ]:
def render_case_timeslices(case, window=6, max_slices=None):
    case_dir = ROOT / case
    slices = align_timeslices(parse_frames(case_dir / 'logs.txt'), window=window)
    if max_slices:
        slices = slices[:max_slices]
    rows = len(slices)
    cols = len(SENSORS)
    if rows == 0:
        print(f'{case}: no frames'); return
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.4, rows * 2.6), squeeze=False)
    for r, slc in enumerate(slices):
        for c, (sensor, label) in enumerate(SENSORS):
            ax = axes[r][c]
            img_path = case_dir / f'{sensor}.jpg'
            if sensor in slc['by_sensor']:
                dets = slc['by_sensor'][sensor]
                title = f't={fmt_clock(int(slc["ts"]))} | {label} | {len(dets)} det'
                draw_detections(ax, img_path, dets, title, show_count=False)
            else:
                draw_detections(ax, img_path, [], f't={fmt_clock(int(slc["ts"]))} | {label} | —')
    fig.suptitle(f'{case} — per-timeslice (window={window}s, {rows} slices)', fontsize=12, y=1.0)
    fig.tight_layout()
    plt.show()

for case in CASES:
    render_case_timeslices(case, window=6)